# Tarefa 5 — Árvore final e teste único (ICMPMASTER)



## 1. Configuração e travas

In [1]:
import json, warnings
from pathlib import Path
import joblib, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, f1_score, recall_score, precision_recall_fscore_support
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings("ignore")

ENTRADA = Path("../../../data/processed")
SAIDA4 = Path("saida_tarefa4")
SAIDA = Path("saida_tarefa5"); SAIDA.mkdir(exist_ok=True)

CLASSES = ["OK", "RISCO", "FALHA"]
SENTINELA = -999
H = 3
SEED = 42

cfg = json.loads((SAIDA4 / "config_modelo.json").read_text(encoding="utf-8"))
COLS = cfg["features"]
proibidas = {"classe", "fluxo_id", "rtt_ms", "rtt_medio_ms", "mediana", "rota_id", "dst_addr", "pais"}
assert not set(COLS) & proibidas, "feature proibida!"
print("Parâmetros da Tarefa 4:", {k: v for k, v in cfg.items() if k != "features"})
print("Colunas:", COLS)

Parâmetros da Tarefa 4: {'criterio': 'gini', 'max_depth': 8, 'min_samples_leaf': 1, 'ccp_alpha': 0.0005, 'seed': 42, 'horizonte_rodadas': 3, 'horizonte_min': 12, 'f1_macro_val': 0.7143, 'recall_falha_val': 0.6825234789710086, 'recall_risco_val': 0.3421793325291516}
Colunas: ['z_robusto', 'aumento_pct', 'jitter_relativo', 'perda_pct', 'timeout_atual', 'n5_timeout', 'n5_aumento80', 'n5_risco', 'z_delta1', 'z_media3_ant', 'z_inclin5']


## 2. Dados e alvo a 12 min (código idêntico ao da Tarefa 4)

In [3]:
d = pd.read_csv(ENTRADA / "dataset_rotulado_B.csv", parse_dates=["timestamp"])
d = d[d["classe"].isin(CLASSES)].copy()          # baseline_insuficiente não tem classe
d = d.sort_values(["fluxo_id", "timestamp"]).reset_index(drop=True)
g = d.groupby("fluxo_id")["z_robusto"]
z = {k: g.shift(k) for k in range(1, 5)}
d["z_delta1"] = d["z_robusto"] - z[1]
lag3 = pd.concat([z[1], z[2], z[3]], axis=1)
d["z_media3_ant"] = lag3.mean(axis=1).where(lag3.notna().sum(axis=1) >= 2)
cinco = (-2 * z[4] - z[3] + z[1] + 2 * d["z_robusto"]) / 10
d["z_inclin5"] = cinco.where(z[2].notna() & z[1].notna() & z[3].notna() & z[4].notna())

gc = d.groupby("fluxo_id")
d["y_fut"] = gc["classe"].shift(-H)
dt = (gc["timestamp"].shift(-H) - d["timestamp"]).dt.total_seconds()
mesmo_bloco = gc["particao"].shift(-H) == d["particao"]
d["tem_alvo"] = mesmo_bloco & dt.between(660, 780)

a = d[d["tem_alvo"]]
tr = a[a["particao"] == "treino"]
te = a[a["particao"] == "teste"].copy()
X = lambda df: df[COLS].fillna(SENTINELA).to_numpy()
print("Teste: N =", len(te), "| fluxos:", te["fluxo_id"].nunique(), "| de", te["timestamp"].min(), "a", te["timestamp"].max())
print(te["y_fut"].value_counts().reindex(CLASSES))

Teste: N = 47890 | fluxos: 65 | de 2026-09-13 19:26:51+00:00 a 2026-09-20 04:19:56+00:00
y_fut
OK       37438
RISCO     3242
FALHA     7210
Name: count, dtype: int64


## 3. Árvore que será medida

In [4]:
modelo = joblib.load(SAIDA4 / "arvore_ajustada.joblib")
assert modelo.n_features_in_ == len(COLS)
print("Critério:", modelo.criterion, "| max_depth:", modelo.max_depth, "| min_samples_leaf:", modelo.min_samples_leaf,
      "| ccp_alpha:", modelo.ccp_alpha, "| profundidade obtida:", modelo.get_depth(), "| folhas:", modelo.get_n_leaves(), "| semente:", modelo.random_state)

def regras(m, cols):
    t, out = m.tree_, []
    def desce(no, cond):
        if t.children_left[no] == -1:
            v = t.value[no][0]; k = int(v.argmax())
            out.append((cond, m.classes_[k], int(t.n_node_samples[no]), v[k] / v.sum())); return
        nome, lim = cols[t.feature[no]], t.threshold[no]
        desce(t.children_left[no], cond + [f"{nome} <= {lim:.3f}"])
        desce(t.children_right[no], cond + [f"{nome} > {lim:.3f}"])
    desce(0, []); return sorted(out, key=lambda r: -r[2])
R = regras(modelo, COLS)
print("\nMaior folha de cada classe (conferir com as regras do diário da Tarefa 4):")
for c in CLASSES:
    r = next((r for r in R if r[1] == c), None)
    if r: print(f"- {c}: se {' e '.join(r[0])} (n={r[2]}, pureza={r[3]:.0%})")

Critério: gini | max_depth: 8 | min_samples_leaf: 1 | ccp_alpha: 0.0005 | profundidade obtida: 8 | folhas: 20 | semente: 42

Maior folha de cada classe (conferir com as regras do diário da Tarefa 4):
- OK: se z_robusto <= 4.008 e n5_risco <= 3.500 e n5_aumento80 <= 0.500 e n5_timeout <= 0.500 e n5_risco <= 1.500 e z_delta1 > -2.408 e z_robusto <= 3.062 e n5_risco <= 0.500 (n=38596, pureza=96%)
- RISCO: se z_robusto <= 4.008 e n5_risco > 3.500 e n5_risco > 4.500 (n=1507, pureza=91%)
- FALHA: se z_robusto > 4.008 e z_media3_ant > 2.488 e z_inclin5 <= 0.990 (n=7868, pureza=96%)


## 4. Teste único (rodar esta célula UMA vez)

In [5]:
te["pred"] = modelo.predict(X(te))          # <-- única medição do teste

cm = pd.DataFrame(confusion_matrix(te["y_fut"], te["pred"], labels=CLASSES),
                  index=[f"real_{c}" for c in CLASSES], columns=[f"prev_{c}" for c in CLASSES])
display(cm); cm.to_csv(SAIDA / "matriz_teste.csv")

p, r, f, s = precision_recall_fscore_support(te["y_fut"], te["pred"], labels=CLASSES, zero_division=0)
rel = pd.DataFrame({"precisao": p, "recall": r, "f1": f, "suporte": s}, index=CLASSES).round(4)
display(rel); rel.to_csv(SAIDA / "relatorio_teste.csv")
f1_macro = f1_score(te["y_fut"], te["pred"], labels=CLASSES, average="macro", zero_division=0)
print(f"F1 macro no teste: {f1_macro:.4f} | recall de FALHA: {r[2]:.4f}")

rf_risco_falha = cm.loc["real_RISCO", "prev_FALHA"], cm.loc["real_FALHA", "prev_RISCO"]
print("Troca RISCO->FALHA:", rf_risco_falha[0], "| FALHA->RISCO:", rf_risco_falha[1],
      "| FALHA->OK:", cm.loc["real_FALHA", "prev_OK"])

,prev_OK,prev_RISCO,prev_FALHA
real_OK,36535,240,663
real_RISCO,2179,744,319
real_FALHA,2169,88,4953


,precisao,recall,f1,suporte
OK,0.8936,0.9759,0.9330,37438
RISCO,0.6940,0.2295,0.3449,3242
FALHA,0.8345,0.6870,0.7536,7210


F1 macro no teste: 0.6772 | recall de FALHA: 0.6870
Troca RISCO->FALHA: 319 | FALHA->RISCO: 88 | FALHA->OK: 2169


## 5. Comparação com “repetir a classe atual” (obrigatória: o alvo é o estado a 12 min)

Mesma amostra de teste, previsão = classe da rodada atual. Se a árvore não ganha, o entregável é um **detector**, não um preditor.

In [6]:
base = te["classe"]
f1_base = f1_score(te["y_fut"], base, labels=CLASSES, average="macro", zero_division=0)
rec_base = dict(zip(CLASSES, recall_score(te["y_fut"], base, labels=CLASSES, average=None, zero_division=0)))
comp = pd.DataFrame({"F1 macro": [f1_macro, f1_base],
                     "Recall FALHA": [r[2], rec_base["FALHA"]], "Recall RISCO": [r[1], rec_base["RISCO"]]},
                    index=["Árvore (Tarefa 4)", "Repetir classe atual"]).round(4)
display(comp); comp.to_csv(SAIDA / "comparacao_persistencia_teste.csv")
print("Ganho de F1 macro da árvore sobre a persistência:", round(f1_macro - f1_base, 4))

,F1 macro,Recall FALHA,Recall RISCO
Árvore (Tarefa 4),0.6772,0.6870,0.2295
Repetir classe atual,0.6695,0.7143,0.3834


Ganho de F1 macro da árvore sobre a persistência: 0.0077


## 6. Casos: acerto em caminho longo estável (OK) e erro relevante

In [7]:
ficha = pd.read_csv(ENTRADA / "baseline_por_fluxo.csv")
print("Colunas da ficha:", list(ficha.columns))
corte_med = ficha["mediana"].median()
te = te.merge(ficha[["fluxo_id", "mediana"]], on="fluxo_id", how="left")
te["caminho"] = np.where(te["mediana"] >= corte_med, "longo", "curto")   # só descrição; não entra na árvore

# (a) acerto em caminho longo estável: fluxo longo com poucas rodadas fora de OK
est = te.groupby("fluxo_id").agg(caminho=("caminho", "first"), n=("y_fut", "size"), ok=("y_fut", lambda s: (s == "OK").mean()))
cand = est[(est["caminho"] == "longo") & (est["ok"] >= 0.95)].sort_values("n", ascending=False)
print("Fluxos longos estáveis candidatos:\n", cand.head())
if len(cand):
    fl = cand.index[0]; b = te[te["fluxo_id"] == fl]
    print(f"\nCASO ACERTO: {fl} | {len(b)} rodadas | acerto {(b['y_fut']==b['pred']).mean():.3f} | "
          f"previsto OK em {(b['pred']=='OK').mean():.1%}")
    display(b[["fluxo_id", "timestamp", "z_robusto", "classe", "y_fut", "pred"]].head(3))

# (b) erro relevante: FALHA real prevista como OK (o mais caro no RFC)
err = te[(te["y_fut"] == "FALHA") & (te["pred"] == "OK")].sort_values("z_robusto", ascending=False)
print(f"\nFALHA prevista como OK: {len(err)}")
display(err[["fluxo_id", "timestamp", "caminho", "z_robusto", "n5_risco", "z_media3_ant", "classe", "y_fut", "pred"]].head(5))

Colunas da ficha: ['fluxo_id', 'n_medicoes_A', 'n_rtt_validos_A', 'mediana', 'MAD', 'iqr', 'jitter_tipico', 'perda_tipica', 'prop_resposta', 'status']
Fluxos longos estáveis candidatos:
                       caminho    n        ok
fluxo_id                                    
1004421|103.77.105.3    longo  753  0.993360
1015084|63.222.187.5    longo  753  0.996016
18087|103.77.105.3      longo  753  0.992032
6358|148.163.220.5      longo  753  0.977424
35681|143.255.248.149   longo  752  0.997340

CASO ACERTO: 1004421|103.77.105.3 | 753 rodadas | acerto 0.993 | previsto OK em 100.0%


,fluxo_id,timestamp,z_robusto,classe,y_fut,pred
5271,1004421|103.77.105.3,2026-09-18 02:08:44+00:00,0.136286,OK,OK,OK
5272,1004421|103.77.105.3,2026-09-18 02:12:41+00:00,-0.750261,OK,OK,OK
5273,1004421|103.77.105.3,2026-09-18 02:16:32+00:00,0.719510,OK,OK,OK



FALHA prevista como OK: 2169


,fluxo_id,timestamp,caminho,z_robusto,n5_risco,z_media3_ant,classe,y_fut,pred
23665,50126|103.77.105.3,2026-09-18 19:00:43+00:00,longo,281.921899,1,-2.258461,FALHA,FALHA,OK
23611,50126|103.77.105.3,2026-09-18 15:24:42+00:00,longo,281.741334,1,-2.210577,FALHA,FALHA,OK
43438,7324|87.238.48.84,2026-09-18 04:53:27+00:00,curto,203.743816,1,0.443584,FALHA,FALHA,OK
2094,1002564|63.222.187.5,2026-09-19 17:22:55+00:00,longo,101.959363,0,-4.873404,FALHA,FALHA,OK
2062,1002564|63.222.187.5,2026-09-19 15:14:56+00:00,longo,95.909769,1,-1.539707,FALHA,FALHA,OK


## 7. Fluxo que a árvore não viu

Na Tarefa 2 o corte é por fluxo no tempo, então **todo fluxo do dataset aparece no treino da árvore oficial**. Para ter fluxo inédito, separamos `fluxo_id` inteiros, retreinamos uma **árvore auxiliar** com **os mesmos hiperparâmetros** só nos fluxos restantes e medimos nos fluxos separados. Não é a árvore oficial e isso deve ser dito no diário. A ficha (mediana e MAD) de cada fluxo já vem só do Período A (piso de 1.500 RTT válidos). Fluxos sem ficha suficiente não têm classe e ficam de fora.

In [8]:
N_FLUXOS_FORA = 6
rng = np.random.default_rng(SEED)
fluxos = sorted(d["fluxo_id"].unique())
print("Fluxos com ficha suficiente:", len(fluxos))

if len(fluxos) < N_FLUXOS_FORA + 10:
    print("Poucos fluxos com baseline suficiente: declarar a limitação no diário e não forçar a conta.")
else:
    # sorteio estratificado simples: precisa de pelo menos 1 fluxo longo e 1 curto
    fm = ficha.set_index("fluxo_id")["mediana"]
    longos = [f for f in fluxos if fm.get(f, np.nan) >= corte_med]; curtos = [f for f in fluxos if fm.get(f, np.nan) < corte_med]
    fora = list(rng.choice(longos, N_FLUXOS_FORA // 2, replace=False)) + list(rng.choice(curtos, N_FLUXOS_FORA // 2, replace=False))
    print("Fluxos separados:", fora)

    treino_aux = a[(a["particao"] == "treino") & ~a["fluxo_id"].isin(fora)]
    aux = DecisionTreeClassifier(criterion=cfg["criterio"], max_depth=cfg["max_depth"], min_samples_leaf=cfg["min_samples_leaf"],
                                 ccp_alpha=cfg["ccp_alpha"], random_state=SEED).fit(X(treino_aux), treino_aux["y_fut"])
    novo = a[a["fluxo_id"].isin(fora)].copy()          # trecho inteiro do Período B desses fluxos
    assert not set(novo["fluxo_id"]) & set(treino_aux["fluxo_id"])
    novo["pred"] = aux.predict(X(novo))
    novo = novo.merge(ficha[["fluxo_id", "mediana"]], on="fluxo_id", how="left")
    novo["caminho"] = np.where(novo["mediana"] >= corte_med, "longo", "curto")

    res = novo.groupby(["fluxo_id", "caminho"]).apply(lambda b: pd.Series({
        "N": len(b), "real_OK_%": (b["y_fut"] == "OK").mean() * 100, "prev_OK_%": (b["pred"] == "OK").mean() * 100,
        "FALHA_real": int((b["y_fut"] == "FALHA").sum()), "FALHA_acertada": int(((b["y_fut"] == "FALHA") & (b["pred"] == "FALHA")).sum()),
        "RISCO_real": int((b["y_fut"] == "RISCO").sum()), "RISCO_acertado": int(((b["y_fut"] == "RISCO") & (b["pred"] == "RISCO")).sum()),
        "acerto": (b["y_fut"] == b["pred"]).mean()})).round(3)
    display(res); res.to_csv(SAIDA / "fluxo_novo.csv")
    print("F1 macro agregado nos fluxos novos:", round(f1_score(novo["y_fut"], novo["pred"], labels=CLASSES, average="macro", zero_division=0), 4))
    display(pd.DataFrame(confusion_matrix(novo["y_fut"], novo["pred"], labels=CLASSES),
                         index=[f"real_{c}" for c in CLASSES], columns=[f"prev_{c}" for c in CLASSES]))

Fluxos com ficha suficiente: 65
Fluxos separados: [np.str_('6380|87.238.48.84'), np.str_('1003574|143.255.248.149'), np.str_('6723|87.238.48.84'), np.str_('1002841|63.222.187.5'), np.str_('6544|87.238.48.84'), np.str_('7020|2001:503:c27::2:30')]


,,N,real_OK_%,prev_OK_%,FALHA_real,FALHA_acertada,RISCO_real,RISCO_acertado,acerto
fluxo_id,caminho,,,,,,,,
1002841|63.222.187.5,curto,2511.0,98.805,100.000,5.0,0.0,25.0,0.0,0.988
1003574|143.255.248.149,longo,2511.0,21.705,58.821,1690.0,879.0,276.0,3.0,0.542
6380|87.238.48.84,longo,2503.0,85.697,99.081,181.0,2.0,177.0,0.0,0.850
6544|87.238.48.84,curto,2503.0,88.654,99.600,173.0,2.0,111.0,0.0,0.884
6723|87.238.48.84,longo,2503.0,89.333,98.841,49.0,14.0,218.0,3.0,0.896
7020|2001:503:c27::2:30,curto,2485.0,96.982,99.920,1.0,0.0,74.0,0.0,0.969


F1 macro agregado nos fluxos novos: 0.4998


,prev_OK,prev_RISCO,prev_FALHA
real_OK,11929,20,87
real_RISCO,794,6,81
real_FALHA,1195,7,897


## 8. Registro final

In [9]:
out = dict(f1_macro_teste=float(f1_macro), recall_falha_teste=float(r[2]), f1_macro_persistencia=float(f1_base),
           n_teste=int(len(te)), parametros={k: v for k, v in cfg.items() if k != "features"}, features=COLS)
(SAIDA / "resultado_teste.json").write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
print("Arquivos:", sorted(p.name for p in SAIDA.iterdir()))

Arquivos: ['comparacao_persistencia_teste.csv', 'fluxo_novo.csv', 'matriz_teste.csv', 'relatorio_teste.csv', 'resultado_teste.json']
